# 01 Data pipeline

Freeze the FRED DEXUSEU snapshot and build past-only features, forward variance targets, chronological splits, and manifests.

## Setup

In [1]:
# Colab setup: clone the private repo, install deps, mount Drive for artifacts.
# Requires a Colab Secret named GITHUB_TOKEN (fine-grained, Contents read-only on this repo).
import base64, os, subprocess, sys
from google.colab import drive, userdata

REPO = "BerkayPolat-lab/gru-currency-volatility-distillation"
REPO_DIR = "/content/gru-currency-volatility-distillation"

def git(*args):
    # Pass the token as an HTTP header through env vars so it never appears in argv,
    # exception messages, or .git/config, and redact it from any error output.
    token = userdata.get("GITHUB_TOKEN").strip()
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
           "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}"}
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        stderr = result.stderr.replace(token, "***").replace(basic, "***")
        raise RuntimeError(f"git {args[0]} failed:\n{stderr}") from None

if not os.path.exists(REPO_DIR):
    git("clone", "-q", f"https://github.com/{REPO}.git", REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/model/requirements.txt"], check=True)
sys.path.insert(0, f"{REPO_DIR}/model/src")
os.chdir(f"{REPO_DIR}/model")

drive.mount("/content/drive")
ARTIFACTS = "/content/drive/MyDrive/gru_vol_artifacts"
os.makedirs(ARTIFACTS, exist_ok=True)

Mounted at /content/drive


In [2]:
import yaml

with open("configs/base.yaml") as f:
    CFG = yaml.safe_load(f)
CFG

{'data': {'source': 'FRED',
  'series_id': 'DEXUSEU',
  'start': '2000-01-01',
  'end': '2025-12-31',
  'snapshot_path': 'data/raw/DEXUSEU.csv',
  'interpolate_missing': False},
 'features': {'window': 60,
  'trailing_vol_windows': [5, 20],
  'inputs': ['log_return', 'squared_return', 'vol_5', 'vol_20']},
 'target': {'horizon': 5, 'scaling': 'divide_by_train_mean'},
 'splits': {'train': ['2000-01-01', '2017-12-31'],
  'validation': ['2018-01-01', '2020-12-31'],
  'test': ['2021-01-01', '2025-12-31'],
  'drop_examples_crossing_split': True},
 'models': {'teacher': {'num_layers': 3, 'hidden_size': 128, 'dropout': 0.2},
  'student': {'num_layers': 1, 'hidden_size': 32, 'dropout': 0.0},
  'output_transform': 'softplus'},
 'historical_control': {'window': 20},
 'training': {'optimizer': 'adamw',
  'learning_rate': 0.001,
  'weight_decay': 0.0001,
  'batch_size': 32,
  'max_epochs': 100,
  'early_stopping_metric': 'val_mae',
  'early_stopping_patience': 10,
  'seeds': [0, 1, 2]},
 'distillat

## Download and freeze the snapshot

Download DEXUSEU for 2000-2025, save a dated CSV, and record its hash in the manifest.

In [11]:
import pandas as pd

exchange_rate = (pd.read_csv(CFG["data"]["snapshot_path"], parse_dates=["observation_date"], index_col="observation_date")["DEXUSEU"].loc[CFG["data"]["start"]:CFG["data"]["end"]].dropna())

print(exchange_rate)

observation_date
2000-01-03    1.0155
2000-01-04    1.0309
2000-01-05    1.0335
2000-01-06    1.0324
2000-01-07    1.0294
               ...  
2025-12-24    1.1780
2025-12-26    1.1766
2025-12-29    1.1756
2025-12-30    1.1765
2025-12-31    1.1736
Name: DEXUSEU, Length: 6518, dtype: float64


## Clean and compute log returns

Keep only observed dates with no interpolation, then compute `r(t) = log(P(t) / P(t-1))`.

In [13]:
import numpy as np

log_returns = np.log(exchange_rate).diff().dropna().rename("log_return")
squared_returns = (log_returns ** 2).rename("squared_return")

In [21]:
print(f"{log_returns.name} max: {log_returns.max():.6f} on {log_returns.idxmax().date()}")
print(f"{log_returns.name} min: {log_returns.min():.6f} on {log_returns.idxmin().date()}")

log_return max: 0.046208 on 2009-03-19
log_return min: -0.030031 on 2008-12-19


In [23]:
rms = np.sqrt((squared_returns).mean())
print(f"{log_returns.name} RMS: {rms:.6f}")

log_return RMS: 0.005795


## Input features

Returns, squared returns, and trailing 5- and 20-observation volatility, all ending at the forecast origin.

In [27]:
vol_5 = np.sqrt(squared_returns.rolling(5).mean()).rename("vol_5")
vol_20 = np.sqrt(squared_returns.rolling(20).mean()).rename("vol_20")

print(vol_5.head())

observation_date
2000-01-04       NaN
2000-01-05       NaN
2000-01-06       NaN
2000-01-07       NaN
2000-01-10    0.0072
Name: vol_5, dtype: float64


In [28]:
print(vol_20.head(20))

observation_date
2000-01-04         NaN
2000-01-05         NaN
2000-01-06         NaN
2000-01-07         NaN
2000-01-10         NaN
2000-01-11         NaN
2000-01-12         NaN
2000-01-13         NaN
2000-01-14         NaN
2000-01-18         NaN
2000-01-19         NaN
2000-01-20         NaN
2000-01-21         NaN
2000-01-24         NaN
2000-01-25         NaN
2000-01-26         NaN
2000-01-27         NaN
2000-01-28         NaN
2000-01-31         NaN
2000-02-01    0.006808
Name: vol_20, dtype: float64


In [30]:
features = pd.concat([log_returns, squared_returns, vol_5, vol_20], axis=1).dropna()
print(features)

                  log_return  squared_return     vol_5    vol_20
observation_date                                                
2000-02-01         -0.002668    7.119880e-06  0.008080  0.006808
2000-02-02          0.003795    1.440256e-05  0.008147  0.005979
2000-02-03          0.012109    1.466285e-04  0.008132  0.006539
2000-02-04         -0.012928    1.671426e-04  0.008197  0.007146
2000-02-07          0.002354    5.540304e-06  0.008256  0.007135
...                      ...             ...       ...       ...
2025-12-24          0.000340    1.153392e-07  0.001685  0.002900
2025-12-26         -0.001189    1.414106e-06  0.001612  0.002818
2025-12-29         -0.000850    7.229555e-07  0.001628  0.002819
2025-12-30          0.000765    5.856437e-07  0.001243  0.002790
2025-12-31         -0.002468    6.090932e-06  0.001336  0.002796

[6498 rows x 4 columns]


## Targets

`v(t,h) = (1/h) * sum r(t+j)^2` for `j = 1..h`, with `h = 5`.

In [ ]:
# TODO

## Chronological splits

Train 2000-2017, validate 2018-2020, test 2021-2025. Drop examples whose forward target crosses a split boundary.

In [ ]:
# TODO

## Scaling

Fit input scaling on train only. Divide targets by the positive train-period mean without centering.

In [ ]:
# TODO

## Save manifests

Save split, feature, and target manifests plus scaling constants to `artifacts/manifests`.

In [ ]:
# TODO